# HiC-ECC | Module 3: Call
Loop and TAD calling on enhanced `.cool` files.

## Config
**Edit only this cell.**

In [ ]:
import yaml

with open('../../config/config.yaml') as f:
    cfg = yaml.safe_load(f)

TISSUES     = cfg['samples']
RESOLUTION  = cfg['resolution']
COOL_DIR    = f"{cfg['output_dir']}/cool_files/{RESOLUTION}"
COOL_SUFFIX = f"_deephic.{RESOLUTION//1000}kb.cool"
OUT_DIR     = f"{cfg['output_dir']}/call_output"
THREADS     = cfg['calling']['loops']['threads']

## Setup

In [ ]:
import os, subprocess

os.makedirs(OUT_DIR, exist_ok=True)
print("Output directory ready:", OUT_DIR)

## Loop Calling (Chromosight)

In [ ]:
for tissue in TISSUES:
    cool = f"{COOL_DIR}/{tissue}/{tissue}{COOL_SUFFIX}"
    out  = f"{OUT_DIR}/{tissue}_loops"

    if not os.path.isfile(cool):
        print(f"[SKIP] Missing file: {cool}")
        continue

    print(f"[loops] {tissue}")
    subprocess.run(
        ["chromosight", "detect", "-t", str(THREADS), cool, out],
        check=True
    )

print("Loop calling done.")

## TAD Calling (hicFindTADs)

In [ ]:
for tissue in TISSUES:
    cool   = f"{COOL_DIR}/{tissue}/{tissue}{COOL_SUFFIX}"
    prefix = f"{OUT_DIR}/{tissue}_tads"

    if not os.path.isfile(cool):
        print(f"[SKIP] Missing file: {cool}")
        continue

    print(f"[TADs] {tissue}")
    subprocess.run(
        ["hicFindTADs", "-m", cool,
         "--outPrefix", prefix,
         "--correctForMultipleTesting", "fdr"],
        check=True
    )

print("TAD calling done.")